In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
df = pd.read_csv("insurance_data.csv")
df.head()

,age,affordability,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,1,1
4,46,1,1


In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df[['age','affordability']],df.bought_insurance,test_size=0.2,random_state=25)

In [6]:
len(X_train)

20

In [7]:
X_train_scaled= X_train.copy()
X_train_scaled['age']= X_train_scaled['age']/100

X_test_scaled= X_test.copy()
X_test_scaled['age']= X_test_scaled['age']/100



In [8]:
X_train_scaled

,age,affordability
0,0.22,1
13,0.49,1
6,0.60,1
17,0.19,1
23,0.50,1
19,0.21,1
24,0.54,0
16,0.58,1
20,0.26,0
3,0.52,1


In [18]:

model = keras.Sequential([
    keras.layers.Dense(
        1,
        input_shape=(2,),       
        activation='sigmoid',
        kernel_initializer='ones',
        bias_initializer='zeros'
    )
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train_scaled,y_train,epochs=500)


Epoch 1/500
1/1 [==============================] - 0s 374ms/step - loss: 0.7634 - accuracy: 0.4000
Epoch 2/500
1/1 [==============================] - 0s 10ms/step - loss: 0.7629 - accuracy: 0.4000
Epoch 3/500
1/1 [==============================] - 0s 6ms/step - loss: 0.7624 - accuracy: 0.4000
Epoch 4/500
1/1 [==============================] - 0s 4ms/step - loss: 0.7618 - accuracy: 0.4000
Epoch 5/500
1/1 [==============================] - 0s 5ms/step - loss: 0.7613 - accuracy: 0.4000
Epoch 6/500
1/1 [==============================] - 0s 0s/step - loss: 0.7608 - accuracy: 0.4000
Epoch 7/500
1/1 [==============================] - 0s 9ms/step - loss: 0.7603 - accuracy: 0.4000
Epoch 8/500
1/1 [==============================] - 0s 3ms/step - loss: 0.7598 - accuracy: 0.4000
Epoch 9/500
1/1 [==============================] - 0s 7ms/step - loss: 0.7593 - accuracy: 0.4000
Epoch 10/500
1/1 [==============================] - 0s 5ms/step - loss: 0.7588 - accuracy: 0.4000
Epoch 11/500
1/1 [=========

In [19]:
model.evaluate(X_test_scaled,y_test)

1/1 [==============================] - 0s 177ms/step - loss: 0.5623 - accuracy: 0.8333


[0.562300443649292, 0.8333333134651184]

In [20]:
model.predict(X_test_scaled) #more than 0.5 means he will buy the insurance

1/1 [==============================] - 0s 129ms/step


array([[0.65371585],
       [0.62825775],
       [0.64442956],
       [0.43246832],
       [0.6641867 ],
       [0.41965082]], dtype=float32)

In [21]:
X_test_scaled


,age,affordability
2,0.47,1
10,0.28,1
21,0.40,1
11,0.27,0
14,0.55,1
9,0.18,0


In [23]:
coef,intercept = model.get_weights() #using tensorflow function!!!
coef,intercept

(array([[0.58243793],
        [0.7907129 ]], dtype=float32),
 array([-0.42904574], dtype=float32))

In [30]:
import math
def sigmoid(x):
    return 1/(1+math.exp(-x))


In [31]:
def prediction_function(age,affordability):
    weighted_sum= coef[0]*age+ coef[1]*affordability +intercept
    return sigmoid(weighted_sum)

In [32]:
prediction_function(0.47,1)

C:\Users\Dell\AppData\Local\Temp\ipykernel_27628\4182413728.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return 1/(1+math.exp(-x))


0.6537158262135281

In [33]:

def sigmoid_numpy(X):
   return 1/(1+np.exp(-X))

sigmoid_numpy(np.array([12,0,1]))

array([0.99999386, 0.5       , 0.73105858])

In [34]:
def log_loss(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i,epsilon) for i in y_predicted]
    y_predicted_new = [min(i,1-epsilon) for i in y_predicted_new]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_true*np.log(y_predicted_new)+(1-y_true)*np.log(1-y_predicted_new))

In [35]:
def gradient_descent(age, affordability, y_true, epochs, loss_thresold):
    w1 = w2 = 1
    bias = 0
    rate = 0.5
    n = len(age)
    for i in range(epochs):
        weighted_sum = w1 * age + w2 * affordability + bias
        y_predicted = sigmoid_numpy(weighted_sum)
        loss = log_loss(y_true, y_predicted)

        w1d = (1/n)*np.dot(np.transpose(age),(y_predicted-y_true)) 
        w2d = (1/n)*np.dot(np.transpose(affordability),(y_predicted-y_true)) 

        bias_d = np.mean(y_predicted-y_true)
        w1 = w1 - rate * w1d
        w2 = w2 - rate * w2d
        bias = bias - rate * bias_d

        print (f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')

        if loss<=loss_thresold:
            break

    return w1, w2, bias

In [42]:
gradient_descent(X_train_scaled['age'],X_train_scaled['affordability'],y_train,4000, 0.3651)


Epoch:0, w1:0.9417429336237633, w2:0.9598996358150476, bias:-0.16004975281869202, loss:0.7633872499715488
Epoch:1, w1:0.8915589178332827, w2:0.9311342163772272, bias:-0.29987918139400804, loss:0.7062511850385205
Epoch:2, w1:0.848830638420335, w2:0.9130339023510808, bias:-0.4211305148209427, loss:0.663699671036478
Epoch:3, w1:0.8127364929867725, w2:0.9045014311242385, bias:-0.5259238600843001, loss:0.6324246470332372
Epoch:4, w1:0.7823807208673199, w2:0.9042322918491189, bias:-0.6165357393850078, loss:0.6094664629237819
Epoch:5, w1:0.7568891902099442, w2:0.9108921899734166, bias:-0.6951670662332271, loss:0.5924345564785555
Epoch:6, w1:0.7354653668250947, w2:0.9232303297315658, bias:-0.7638107461094875, loss:0.5795308000606487
Epoch:7, w1:0.7174145482376012, w2:0.9401363212799361, bias:-0.8241970456249407, loss:0.5694671501397951
Epoch:8, w1:0.7021477259459787, w2:0.9606584178928229, bias:-0.8777880239694975, loss:0.5613505511423982
Epoch:9, w1:0.6891744911389016, w2:0.9839994457384778, 

(1.9918507704032924, 4.162915524759508, -4.279711243469651)

In [43]:
coef,intercept

(array([[0.58243793],
        [0.7907129 ]], dtype=float32),
 array([-0.42904574], dtype=float32))